In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("modelTraining").getOrCreate()

In [0]:
feature_df = spark.table("teleco_churn_datalakehouse.gold.customer_churn_features")

feature_df.display()

In [0]:
# Test - remove nulls from feature_df
feature_df = feature_df.dropna(subset=[
    "Tenure_Months",
    "Monthly_Charges",
    "Total_Charges"
])

In [0]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=[
        "Tenure_Months",
        "Monthly_Charges",
        "Total_Charges"
    ],
    outputCol="features"
)

df_features = assembler.transform(feature_df)

- Creating the Train/test split in pyspark

In [0]:
train_df, test_df = df_features.randomSplit([0.8, 0.2], seed=42)

- veryfying the split

In [0]:
print("Training rows:", train_df.count())
print("Test rows:", test_df.count())

- Train the model

In [0]:
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(
    featuresCol="features",
    labelCol="churn_label"
)

model = lr.fit(train_df)

- Making predictions

In [0]:
predictions = model.transform(test_df)

- Evaluating the model

In [0]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

evaluator = BinaryClassificationEvaluator(
    labelCol="churn_label"
)

auc = evaluator.evaluate(predictions)

print("AUC:", auc)

In [0]:
%sql
-- Creating a volume for saving my model
CREATE VOLUME IF NOT EXISTS teleco_churn_datalakehouse.telecom_ml_database.models;

In [0]:
# Saving the model
# The path format is: /Volumes/<catalog>/<schema>/<volume>/<path>
model.write().overwrite().save("/Volumes/teleco_churn_datalakehouse/telecom_ml_database/models/my_model")

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS teleco_churn_datalakehouse.telecom_ml_database ;

In [0]:
# will look into whether the training data should be saved as a table

train_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("teleco_churn_datalakehouse.telecom_ml_database.train_table")

test_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("teleco_churn_datalakehouse.telecom_ml_database.test_table")